# Extremely basic version
issue: even number of errors in one block will not be corrected

In [1]:
import textwrap

In [2]:
alice_bits = '101101100000'
bob_bits = '101001100001'

In [3]:
# helper
def parity(x):
    
    return sum([int(bit) for bit in x]) % 2

# get BER. Cascade uses k = 1/e, where e is the BER and k is number of blocks. For now, pretend BER is 0.25
k = 4

alice_blocks = textwrap.wrap(alice_bits, k)
bob_blocks = textwrap.wrap(bob_bits, k)

# use parity: even number of 1s in the string vs odd number of 1s in the string (results in 0 or 1)
# compare parity blockwise: if mismatch, error is found

In [4]:
# go through blocks
# if error in block, split block in half and check parity again
# when found, flip bit in bob's string

def check_block(a, b, start):
    if len(a) == 1:
        if parity(a) != parity(b):
            return start
        return None

    if parity(a) == parity(b):
        return None

    mid = len(a) // 2

    result = check_block(a[:mid], b[:mid], start)
    if result is not None:
        return result

    return check_block(a[mid:], b[mid:], start + mid)


bob_bits_list = list(bob_bits)

for a, b in zip(alice_blocks, bob_blocks):
    if parity(a) != parity(b):
        idx = check_block(a, b, 0)

        if idx is not None:
            bob_bits_list[idx] = '1' if bob_bits_list[idx] == '0' else '0'

bob_bits_final = ''.join(bob_bits_list)
print("Corrected Bob bits:", bob_bits_final)

Corrected Bob bits: 101001100001


# Cascade

To get cascade-python locally, just run
```
git submodule add https://github.com/brunorijsman/cascade-python.git
```

Then the rest of the code will run

In [6]:
import sys
import os

sys.path.insert(0, os.path.abspath("cascade-python"))

In [7]:
from cascade.key import Key
from cascade.mock_classical_channel import MockClassicalChannel
from cascade.reconciliation import Reconciliation

In [ ]:
# small helper function, as we need to convert the bitstrings to keyobjects
def key_from_bits(bitstring):
    key = Key()
    key._size = len(bitstring)

    for i, b in enumerate(bitstring):
        key._bits[i] = int(b)

    return key

In [22]:
with open('alice_key.txt', 'r') as file:
    correct_bits = file.read()

with open('bob_key.txt', 'r') as file:
    noisy_bits = file.read()

correct_key = key_from_bits(correct_bits)
noisy_key = key_from_bits(noisy_bits)

channel = MockClassicalChannel(correct_key)

algorithm = "original" # there are some others they can be found in cascade/tests/test_algorithm I think, but idk what the difference is

rec = Reconciliation(
    algorithm,
    channel,
    noisy_key,
    0.1
)

rec.reconcile()
rec_key = rec.get_reconciled_key()

print("Correct:", correct_key)
print("Noisy:   ", noisy_key)
print("Repaired:", rec_key)

for a,b in zip(str(correct_key), str(rec_key)):
    if a != b:
        print("Error at position:", correct_key.index(a))

Correct: 1010001011001101000011101000001001111011000111111000111011111001101001001101011101100011100001110001000110110000100101011111010001000011100011000101110100101001110011000100100100010010110110111000100011001011111011111010101101000000001111010000111110011011110001101101010110110110010100010000011010000001010110011011001010111010010101010001110001010101010010111100101110010111000011001101001010100010100111100101110000100110001000100011000010101001110100011000011010011000111101101011100010011110110010011110000110100010010100010101101000110010100111011111111110100101000010010011010100100111110110000111001000110010011110111000001101000001010000101111110011001110111100000011010000110010010100011010100110111011011110011110100100000010000011110111000100011111110001100101111000100000011010000110001101001110100010111101011100111010101101011100000111100101010110001000111001110001011000111000001110001001100111111111011010110100000111110011010000000111001001001110001100000001111101000010111